In [ ]:
#use "../../lib/nb.ml";;

# Converting $\LaTeX$ to <span style="font-variant:small-caps;">Html</span>

The purpose of the following exercise is to implement a translator from [$\LaTeX$](http://www.latex-project.org) to
[MathML](https://www.tutorialspoint.com/mathml/index.htm).  $\LaTeX$ is a document markup language
that is especially well suited to present text that contains mathematical formulas.  MathML is the part of <span style="font-variant:small-caps;">Html</span> that deals with the representation of mathematical formulas.  As $\LaTeX$ provides a very rich
document markup language and we can only afford to spend a few hours on this exercise, we confine
ourselves to a small subset of $\LaTeX$.  The file `example.tex` contains some $\LaTeX$.  The goal of this exercise is to implement a translator that is able to transform this file into MathML.

The translator has to be implemented using nothing but regular expressions, i.e. the module `Re` from the
file `../../lib/regex.ml`, which offers the functions of Python's module `re` on top of the library PCRE2.  In
contrast to the scanner developed in the notebook `01-RegExp-Scanner.ipynb`, this scanner needs two additional
ingredients:
1. It uses **scanner states**, because the rules that apply inside a mathematical formula are
   different from the rules that apply in ordinary text.
2. It uses a **stack**, because once we encounter a closing brace `}` we have to know which
   construct is closed by this brace.  As these constructs can be nested, a single variable is not
   enough to store this information.

We start with reading the file.  The function `read_file` is defined in `../../lib/nb.ml`.

In [ ]:
let data = read_file "example.tex"

Now data contains the text that is stored in this file.

In [ ]:
print_endline data

Let us look at the output file `example.pdf` that is produced if we run $\LaTeX$ on this file. 
If you are not running <span style="font-variant:small-caps;">MacOS</span> you have to replace the command
`open` with the name of an executable that can open a `.pdf`-file.

In [ ]:
Sys.command "open example.pdf"

Next, we open the file `example.html`.  The scanner we are going to implement has to write its output into this file.

The functions defined below write their output with the function `write`.  Later, we want to redirect the
output into a string instead of the file.  Python can assign a `StringIO` object to the variable `outfile`, since
such an object has a method `write`, just like a file.  In OCaml, the variable `outfile` is a *reference* to a
value of the type `sink`: a sink is either a file, i.e. an `out_channel`, or a *buffer*, which collects the output
in memory.  `!outfile` is the current sink and `outfile := ...` replaces it.  The function `write` checks which
kind of sink is used and writes its argument to this sink.

In [ ]:
type sink = File of out_channel | Memory of Buffer.t

let outfile : sink ref = ref (File (open_out "example.html"))

let write (s : string) : unit =
  match !outfile with
  | File oc  -> output_string oc s
  | Memory b -> Buffer.add_string b s

let close_outfile () : unit =
  match !outfile with
  | File oc  -> close_out oc
  | Memory _ -> ()

let contents_of_outfile () : string =
  match !outfile with
  | File _   -> ""
  | Memory b -> Buffer.contents b

<hr style="height:4px;background-color:blue">
Below are some predefined functions that you can use to create the <span style="font-variant:small-caps;">Html</span> file.
<hr style="height:4px;background-color:blue">

The function `start_html` writes the header of the <span style="font-variant:small-caps;">Html</span> file
and the opening `<body>` tag to the file opened above.  The `<script>` tag loads
[MathJax](https://www.mathjax.org), which is the *JavaScript* library that renders the
<span style="font-variant:small-caps;">MathML</span> that we are going to produce.

**Function `start_html ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the header of the <span style="font-variant:small-caps;">Html</span> file, which loads *MathJax*, and the opening tag `<body>` to the current output, i.e. to `!outfile`.

In [ ]:
let start_html () : unit =
  write "<!DOCTYPE html>\n";
  write "<html lang=\"en\">\n";
  write "<head>\n";
  write "<meta charset=\"utf-8\">\n";
  write "<title>Converted LaTeX Document</title>\n";
  write "<script src=\"https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js\" async>\n";
  write "</script>\n";
  write "</head>\n";
  write "<body>\n\n"

The function `end_html` writes the closing `</body>` and `</html>` tags.

**Function `end_html ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the closing tags `</body>` and `</html>` to the current output, i.e. to `!outfile`.

In [ ]:
let end_html () : unit =
  write "</body>\n";
  write "</html>\n"

The function `start_math_block` starts a math block.  This is useful for formulas enclosed in `$$`.  This type of formulas is displayed in a line by itself.

**Function `start_math_block ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the opening tag of a block formula to the current output, i.e. to `!outfile`.

In [ ]:
let start_math_block () : unit =
  write "<math xmlns=\"http://www.w3.org/1998/Math/MathML\" display=\"block\">\n"

The function `start_math_inline` starts an <em style="color:blue">inline formula</em>, i.e. a formula enclosed in `$`.  Formulas of this type are part of the surrounding text.

**Function `start_math_inline ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the opening tag of an inline formula to the current output, i.e. to `!outfile`.

In [ ]:
let start_math_inline () : unit =
  write "<math xmlns=\"http://www.w3.org/1998/Math/MathML\" display=\"inline\">\n"

The function `end_math` ends a math block.

**Function `end_math ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the closing tag `</math>` to the current output, i.e. to `!outfile`.

In [ ]:
let end_math () : unit =
  write "</math>\n"

The functions `start_sum` and `end_sum` write code to display formulas involving sums.  For example, to display  the expression
$$ \sum\limits_{i=1}^n i^2 $$
we can use the following MathML:
```
<munderover>
<mo>&sum;</mo>
<mrow>
<mi>i</mi>
<mo>=</mo>
<mn>1</mn>
</mrow>
<mrow>
<mi>n</mi>
</mrow>
</munderover>
<msup>
<mi>i</mi>
<mrow>
<mn>2</mn>
</mrow>
</msup>

```

**Function `start_sum ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the opening tag `<munderover>` and the sum symbol to the current output, i.e. to `!outfile`.

In [ ]:
let start_sum () : unit =
  write "<munderover>\n";
  write "<mo>&sum;</mo>\n"

**Function `end_sum ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the closing tag `</munderover>` to the current output, i.e. to `!outfile`.

In [ ]:
let end_sum () : unit =
  write "</munderover>\n"

The functions `start_sqrt` and `end_sqrt` write code to display formulas involving square roots.  For example, to display  the expression
$$ \sqrt{a^2 + b^2} $$
we can use the following MathML:
```
<msqrt>
<mrow>
<msup>
<mi>a</mi>
<mrow>
<mn>2</mn>
</mrow>
</msup>
<mo>+</mo>
<msup>
<mi>b</mi>
<mrow>
<mn>2</mn>
</mrow>
</msup>
</mrow>
</msqrt>
```

**Function `start_sqrt ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the opening tag `<msqrt>` to the current output, i.e. to `!outfile`.

In [ ]:
let start_sqrt () : unit =
  write "<msqrt>\n"

**Function `end_sqrt ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the closing tag `</msqrt>` to the current output, i.e. to `!outfile`.

In [ ]:
let end_sqrt () : unit =
  write "</msqrt>\n"

In order to write exponents we have to use the tag `<msup>`.  For example, the expression $a^2$ 
is equivalent to the following markup:
```
<msup>
<mi>a</mi>
<mrow>
<mn>2</mn>
</mrow>
</msup>
```
Note that the exponent is enclosed in `<mrow>` `</mrow>` tags.

**Function `start_super ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the opening tag `<msup>` to the current output, i.e. to `!outfile`.

In [ ]:
let start_super () : unit =
  write "<msup>\n"

**Function `end_super ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the closing tag `</msup>` to the current output, i.e. to `!outfile`.

In [ ]:
let end_super () : unit =
  write "</msup>\n"

In order to write fractions we have to use the tag `<mfrac>`.  For example, the expression $\frac{1}{6}$ 
is equivalent to the following markup:
```
<mfrac>
<mrow>
<mn>1</mn>
</mrow>
<mrow>
<mn>6</mn>
</mrow>
</mfrac>
```
Note that both nominator and denominator are enclosed in `<mrow>` `</mrow>` tags.

**Function `start_fraction ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the opening tag `<mfrac>` to the current output, i.e. to `!outfile`.

In [ ]:
let start_fraction () : unit =
  write "<mfrac>\n"

**Function `end_fraction ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the closing tag `</mfrac>` to the current output, i.e. to `!outfile`.

In [ ]:
let end_fraction () : unit =
  write "</mfrac>\n"

Arguments of functions like the square root or exponents have to be enclosed in pairs of `<mrow>` and `</mrow>` tags.  

**Function `start_row ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the opening tag `<mrow>` to the current output, i.e. to `!outfile`.

In [ ]:
let start_row () : unit =
  write "<mrow>\n"

**Function `end_row ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the closing tag `</mrow>` to the current output, i.e. to `!outfile`.

In [ ]:
let end_row () : unit =
  write "</mrow>\n"

Variable names should be enclosed in pairs of `<mi>` and `</mi>` tags.  For example, the variable $x$ is displayed by the following MathML:
```
<mi>x</mi>
```

**Function `write_var v`**
- *Input:* `v` is the name of a variable.
- *Output:* `()`.  The function writes the variable `v`, enclosed in the tags `<mi>` and `</mi>`, to the current output, i.e. to `!outfile`.

In [ ]:
let write_var (v : string) : unit =
  write ("<mi>" ^ v ^ "</mi>\n")

Numbers should be enclosed in pairs of `<mn>` and `</mn>` tags.  For example, the number $6$ is displayed by the following MathML:
```
<mn>6</mn>
```

**Function `write_number n`**
- *Input:* `n` is a string of digits.
- *Output:* `()`.  The function writes the number `n`, enclosed in the tags `<mn>` and `</mn>`, to the current output, i.e. to `!outfile`.

In [ ]:
let write_number (n : string) : unit =
  write ("<mn>" ^ n ^ "</mn>\n")

The symbol $\cdot$ is created by the following MathML:
```
<mo>&sdot;</mo>
```

**Function `write_times ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the multiplication operator `&sdot;` to the current output, i.e. to `!outfile`.

In [ ]:
let write_times () : unit =
  write "<mo>&sdot;</mo>\n"

Mathematical operators should be enclosed in pairs of `<mo>` and `</mo>` tags.  For example, the operator $+$ is displayed by the following MathML:
```
<mo>+</mo>
```

**Function `write_operator op`**
- *Input:* `op` is an operator symbol like `+`.
- *Output:* `()`.  The function writes the operator `op`, enclosed in the tags `<mo>` and `</mo>`, to the current output, i.e. to `!outfile`.

In [ ]:
let write_operator (op : string) : unit =
  write ("<mo>" ^ op ^ "</mo>\n")

The symbol $\pi$ is created by the following MathML:
```
<mi>&pi;</mi>
```

**Function `write_pi ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the constant $\pi$ to the current output, i.e. to `!outfile`.

In [ ]:
let write_pi () : unit =
  write "<mi>&pi;</mi>\n"

The symbol $\leq$ is created by the following MathML:
```
<mo>&le;</mo>
```

**Function `write_leq ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the operator $\leq$ to the current output, i.e. to `!outfile`.

In [ ]:
let write_leq () : unit =
  write "<mo>&le;</mo>\n"

The symbol $\geq$ is created by the following MathML:
```
<mo>&ge;</mo>
```

**Function `write_geq ()`**
- *Input:* No arguments.
- *Output:* `()`.  The function writes the operator $\geq$ to the current output, i.e. to `!outfile`.

In [ ]:
let write_geq () : unit =
  write "<mo>&ge;</mo>\n"

The function `write_any` writes a single character unadorned to the output file.

**Function `write_any char`**
- *Input:* `char` is a character of ordinary text.
- *Output:* `()`.  The function writes `char` unchanged to the current output, i.e. to `!outfile`.

In [ ]:
let write_any (char : string) : unit =
  write char

<hr style="height:4px;background-color:blue">
The scanner starts here.
<hr style="height:4px;background-color:blue">

## Imports

The only module that we need is the module `Re`.

In [ ]:
load_lib "regex.ml"

## Scanner States

Our scanner has to behave differently in different parts of the input file.  Therefore it is
organized as a [finite state machine](https://en.wikipedia.org/wiki/Finite-state\_machine) that has
the following four states:
- `PREAMBLE` is the state the scanner starts in.  Everything that occurs before the string `begin{document}`
  belongs to the preamble of the $\LaTeX$ document and is discarded.
- `TEXT` is the state that the scanner is in when it reads ordinary text, i.e. text that is *not*
  part of a mathematical formula.  This text is echoed unchanged to the output file.
- `FORMULA` is the state that the scanner is in once it has read either a `$` or a `$$`.  In this
  state the input has to be translated into <span style="font-variant:small-caps;">MathML</span>.
- `DONE` is the state that is reached once the string `\end{document}` has been read.  In this
  state the scanner stops.

For every state we need a separate token specification.  Note that in the state `DONE` the scanner
does not read anything anymore, so no token specification is needed for this state.

In Python, the states, the names of the tokens, and the markers on the stack are strings.  In OCaml, they are
the constructors of the *variant types* `state`, `token`, and `marker` defined below.

**Type safety:** If a token name is misspelled, e.g. `SUM_LIMIT` instead of `SUM_LIMITS`, the compiler
reports an error.  Furthermore, the compiler checks that every `match` expression handles every possible
case.  For example, if one of the markers is not handled when a closing brace is read, the compiler
issues a warning.  In Python, a misspelled string is only noticed when the program behaves strangely.

The functions `name_of_token` and `token_of_name` convert between tokens and the names of the groups in
the master patterns.  The exception `Syntax_error` corresponds to Python's `SyntaxError`.

In [ ]:
type state = PREAMBLE | TEXT | FORMULA | DONE

type token =
  | HEAD | BEGIN_DOCUMENT | SKIP                                    (* tokens of the state PREAMBLE *)
  | END_DOCUMENT | DOLLAR_DOLLAR | DOLLAR | ANY                     (* tokens of the state TEXT *)
  | SUM_LIMITS | SQRT | FRAC | CDOT | LEQ | GEQ | PI | VARIABLE_HAT  (* tokens of the state FORMULA *)
  | NUMBER_HAT | CB_HAT_OB | CB_OB | CB | NUMBER | VARIABLE | OPERATOR | WS | MISMATCH

type marker = Sqrt | Frac_num | Frac_den | Sum_lower | Sum_upper | Super

exception Syntax_error of string

let token_names : (token * string) list =
  [ (HEAD, "HEAD"); (BEGIN_DOCUMENT, "BEGIN_DOCUMENT"); (SKIP, "SKIP"); (END_DOCUMENT, "END_DOCUMENT");
    (DOLLAR_DOLLAR, "DOLLAR_DOLLAR"); (DOLLAR, "DOLLAR"); (ANY, "ANY"); (SUM_LIMITS, "SUM_LIMITS");
    (SQRT, "SQRT"); (FRAC, "FRAC"); (CDOT, "CDOT"); (LEQ, "LEQ"); (GEQ, "GEQ"); (PI, "PI");
    (VARIABLE_HAT, "VARIABLE_HAT"); (NUMBER_HAT, "NUMBER_HAT"); (CB_HAT_OB, "CB_HAT_OB"); (CB_OB, "CB_OB");
    (CB, "CB"); (NUMBER, "NUMBER"); (VARIABLE, "VARIABLE"); (OPERATOR, "OPERATOR"); (WS, "WS");
    (MISMATCH, "MISMATCH") ]

let name_of_token (t : token) : string = List.assoc t token_names
let token_of_name (n : string) : token = fst (List.find (fun (_, m) -> m = n) token_names)

let marker_name (m : marker) : string =
  match m with
  | Sqrt -> "SQRT" | Frac_num -> "FRAC_NUM" | Frac_den -> "FRAC_DEN"
  | Sum_lower -> "SUM_LOWER" | Sum_upper -> "SUM_UPPER" | Super -> "SUPER"

## Your Task

Everything that follows is already prepared for you, with two exceptions:
1. In the function `token_specifications` you have to add the regular expressions for those
   tokens of the state `FORMULA` that are currently commented out.
2. In the function `process_formula` you have to implement all cases that are currently marked with
   `(* your code here *)`.

Everything else &mdash; the functions that write the
<span style="font-variant:small-caps;">MathML</span>, the master patterns, the functions
`process_preamble` and `process_text`, and the function `scan` that drives the scanner &mdash;
is given.  It is a good idea to proceed in small steps and to test your implementation after
every token that you have added.

## Specifying the Tokens

The function `token_specifications` maps every state to the list of token specifications that is
used in this state.  (In Python, this is a dictionary.  A function that uses `match` has the advantage that
the compiler checks that every state is handled.)  As in the notebook `01-RegExp-Scanner.ipynb`, every token
specification is a pair: The first component is the token, while the second component is a regular
expression describing the strings that are recognized as tokens of this type.  The regular expressions are
written as quoted strings `{|...|}`, which correspond to Python's raw strings.

Since PCRE2 always chooses the **first** alternative that matches at the current
position, the order of these pairs is important:
- `VARIABLE_HAT` has to precede `VARIABLE`, for otherwise the string `i^{` would be scanned as the
  variable `i`.
- `CB_HAT_OB` has to precede `CB_OB`, which in turn has to precede `CB`, for otherwise the string
  `}^{` would be scanned as a single closing brace.
- The last specification of the states `PREAMBLE`, `TEXT`, and `FORMULA` matches an arbitrary
  character.  Hence, in these states a match is always found and we never have to deal with the case that
  `Re.Pattern.match_` returns `None`.  In the state `FORMULA` this catch-all token is called `MISMATCH`,
  because any character that is not covered by one of the preceding regular expressions is an
  error, while in the states `PREAMBLE` and `TEXT` arbitrary characters are perfectly acceptable.

In [ ]:
let token_specifications (state : state) : (token * string) list =
  match state with
  | PREAMBLE -> [
      (HEAD,           {|\\documentclass\{article\}|});
      (BEGIN_DOCUMENT, {|\\begin\{document\}|});
      (SKIP,           {|[\s\S]|});                    (* discard everything else *)
    ]
  | TEXT -> [
      (END_DOCUMENT,   {|\\end\{document\}|});
      (DOLLAR_DOLLAR,  {|\$\$|});
      (DOLLAR,         {|\$|});
      (ANY,            {|[\s\S]|});                    (* echo everything else *)
    ]
  | FORMULA -> [
      (DOLLAR_DOLLAR,  {|\$\$|});
      (DOLLAR,         {|\$|});
   (* (SUM_LIMITS,     {|your regexp here|});            matches \sum\limits_{  *)
      (SQRT,           {|\\sqrt\s*\{|});               (* matches \sqrt{ *)
   (* (FRAC,           {|your regexp here|});            matches \frac{  *)
   (* (CDOT,           {|your regexp here|});            matches \cdot  *)
   (* (LEQ,            {|your regexp here|});            matches \leq  *)
   (* (GEQ,            {|your regexp here|});            matches \geq  *)
   (* (PI,             {|your regexp here|});            matches \pi  *)
   (* (VARIABLE_HAT,   {|your regexp here|});            matches e.g. i^{  *)
   (* (NUMBER_HAT,     {|your regexp here|});            matches e.g. 2^{  *)
   (* (CB_HAT_OB,      {|your regexp here|});            matches }^{  *)
   (* (CB_OB,          {|your regexp here|});            matches }{  *)
      (CB,             {|\}|});
      (NUMBER,         {|0|[1-9][0-9]*|});
      (VARIABLE,       {|[a-zA-Z]+|});
   (* (OPERATOR,       {|your regexp here|});            matches a single operator symbol  *)
      (WS,             {|\s+|});                       (* whitespace is not significant *)
      (MISMATCH,       {|[\s\S]|});                    (* anything else is an error *)
    ]
  | DONE -> []

## Building the Master Patterns

For every state we combine the regular expressions of the corresponding token specification into a
single *master pattern*.  As before, every regular expression is wrapped into a *named group* of
the form
```
(?P<NAME>regexp)
```
and these named groups are joined using the alternative operator `|`.  Note that the names of the
groups only have to be unique **within** a single master pattern.  Therefore the token
`DOLLAR_DOLLAR` can occur both in the state `TEXT` and in the state `FORMULA`.

In contrast to the notebook `01-RegExp-Scanner.ipynb` we compile the master patterns via
`Re.compile`.  The reason is that we can not use the function `Re.finditer` here: as the scanner
switches between different patterns, we have to control the position where the next match has to
start ourselves.  This is done via the function `Re.Pattern.match_`, which takes the position as its
optional argument `~pos`.  The function `master_pattern` returns the master pattern of a state.

In [ ]:
let master_patterns : (state * Re.pattern) list =
  List.map
    (fun state ->
      let alternatives =
        List.map (fun (token, regexp) -> Printf.sprintf "(?P<%s>%s)" (name_of_token token) regexp)
                 (token_specifications state)
      in
      (state, Re.compile (String.concat "|" alternatives)))
    [ PREAMBLE; TEXT; FORMULA ]

let master_pattern (state : state) : Re.pattern = List.assoc state master_patterns

Let us take a look at the master pattern that is used inside a formula:

In [ ]:
Re.pattern (master_pattern FORMULA)

## Processing the Tokens of the State `PREAMBLE`

The function `process_preamble` takes a token that has been found in the state `PREAMBLE` and
returns the state that the scanner has to switch to.  Everything that is read in this state is
discarded.  Once the string `\begin{document}` is found, the header of the
<span style="font-variant:small-caps;">Html</span> file is written and the scanner switches to the
state `TEXT`.

**Function `process_preamble token value`**
- *Input:* `token` is a token that has been read in the state `PREAMBLE` and `value` is the string that has been matched.
- *Output:* The next state of the scanner: `TEXT` if `token` is `BEGIN_DOCUMENT`, otherwise `PREAMBLE`.  At the beginning of the document, the header of the <span style="font-variant:small-caps;">Html</span> file is written.

In [ ]:
let process_preamble (token : token) (_value : string) : state =
  match token with
  | BEGIN_DOCUMENT -> start_html (); TEXT
  | _ -> PREAMBLE

## Processing the Tokens of the State `TEXT`

In the state `TEXT` the scanner echoes its input unchanged.  There are three exceptions:
- The string `$$` starts a formula that is displayed in a line of its own.
- The string `$` starts an inline formula.
- The string `\end{document}` ends the document.

**Function `process_text token value`**
- *Input:* `token` is a token that has been read in the state `TEXT` and `value` is the string that has been matched.
- *Output:* The next state of the scanner: `DONE` at the end of the document, `FORMULA` at the start of a formula, and `TEXT` otherwise.  The function writes the corresponding output.

In [ ]:
let process_text (token : token) (value : string) : state =
  match token with
  | END_DOCUMENT  -> end_html ();          DONE
  | DOLLAR_DOLLAR -> start_math_block ();  FORMULA
  | DOLLAR        -> start_math_inline (); FORMULA
  | _             -> write_any value;      TEXT    (* token = ANY *)

## Processing the Tokens of the State `FORMULA`

This is the interesting part.  The problem that we have to solve is the following: Once we
encounter a closing brace `}`, we have to know whether this brace closes the argument of a square
root, the lower or upper limit of a sum, the numerator or the denominator of a fraction, or an
exponent.  As these constructs can be nested, a single variable is not sufficient to store this
information.  Therefore, the function `process_formula` takes a stack `stack` as its third argument.
We use the module `Stack` of the OCaml standard library: `Stack.push x stack` pushes `x` onto the stack, while
`Stack.pop stack` removes the topmost element and returns it.  Every time a construct that has to be closed
by a brace is opened, we push a corresponding marker onto this stack.  The markers that we use are the following:
- `Sqrt`      is pushed when the string `\sqrt{` is read,
- `Frac_num`  is pushed when the string `\frac{` is read, i.e. when the numerator starts,
- `Frac_den`  is pushed when the string `}{` is read, i.e. when the denominator starts,
- `Sum_lower` is pushed when the string `\sum\limits_{` is read,
- `Sum_upper` is pushed when the string `}^{` is read that separates the limits of a sum,
- `Super`     is pushed when a string like `i^{` is read.

When a closing brace is found, the marker on top of the stack is popped and tells us which
<span style="font-variant:small-caps;">MathML</span> tags have to be written.  Note that the
markers `Frac_num` and `Sum_lower` must not be closed by a simple `}`: the numerator of a
fraction is followed by its denominator, while the lower limit of a sum is followed by its upper
limit.

If the stack is not empty when a formula ends, the input contains an unbalanced opening brace and
an exception is raised.

The function `process_formula` below is **incomplete**: every place that is marked with
```
(* your code here *)
```
has to be replaced by your own code.  The token `SQRT` and the marker `Sqrt` are already
implemented and show you how the stack has to be used: when the string `\sqrt{` is read, the
opening tags are written and the marker `Sqrt` is pushed, while the closing brace pops this
marker again and writes the corresponding closing tags.  All other constructs work the same way.

Every case of the `match` expression returns the next state of the scanner.  The tokens of the states
`PREAMBLE` and `TEXT` can not occur in the state `FORMULA`.  Since the compiler insists that every token is
handled, the last case deals with them.

**Function `process_formula token value stack`**
- *Input:* `token` is a token that has been read in the state `FORMULA`, `value` is the string that has been matched, and `stack` is the stack of markers of the constructs that are currently open.  The function modifies `stack`.
- *Output:* The next state of the scanner: `TEXT` at the end of the formula and `FORMULA` otherwise.  The function writes the <span style="font-variant:small-caps;">MathML</span> for the token.  The exception `Syntax_error` is raised if the formula is malformed.

In [ ]:
let process_formula (token : token) (value : string) (stack : marker Stack.t) : state =
  match token with
  | DOLLAR_DOLLAR | DOLLAR ->
      if not (Stack.is_empty stack) then
        raise (Syntax_error (Printf.sprintf "formula ended while %s was still open" (marker_name (Stack.top stack))));
      end_math ();
      TEXT
  | SUM_LIMITS ->
      (* your code here *)
      FORMULA
  | SQRT ->
      start_sqrt ();
      start_row ();
      Stack.push Sqrt stack;
      FORMULA
  | FRAC ->
      (* your code here *)
      FORMULA
  | VARIABLE_HAT | NUMBER_HAT ->
      (* The string that has been matched has the form i^{ or 2^{.  Chop off the
         trailing ^{, write the base, open a row for the exponent and push a marker. *)
      (* your code here *)
      FORMULA
  | CB_HAT_OB ->                                (* the string }^{ *)
      (* your code here *)
      FORMULA
  | CB_OB ->                                    (* the string }{ *)
      (* your code here *)
      FORMULA
  | CB ->                                       (* the string } *)
      if Stack.is_empty stack then
        raise (Syntax_error {|closing brace "}" without a matching opening brace|});
      (match Stack.pop stack with
       | Sqrt ->
           end_row ();
           end_sqrt ()
       | Frac_den ->
           (* your code here *)
           ()
       | Sum_upper ->
           (* your code here *)
           ()
       | Super ->
           (* your code here *)
           ()
       | (Frac_num | Sum_lower) as top ->
           raise (Syntax_error (Printf.sprintf {|"}" can not close %s|} (marker_name top))));
      FORMULA
  | NUMBER   -> write_number value; FORMULA
  | VARIABLE -> write_var value;    FORMULA
  | PI ->
      (* your code here *)
      FORMULA
  | CDOT ->
      (* your code here *)
      FORMULA
  | LEQ ->
      (* your code here *)
      FORMULA
  | GEQ ->
      (* your code here *)
      FORMULA
  | OPERATOR ->
      (* your code here *)
      FORMULA
  | WS -> FORMULA
  | MISMATCH ->
      raise (Syntax_error (Printf.sprintf "unexpected character %s inside a formula" (repr value)))
  | HEAD | BEGIN_DOCUMENT | SKIP | END_DOCUMENT | ANY ->
      invalid_arg ("process_formula: " ^ name_of_token token ^ " is not a token of the state FORMULA")

## The Scanner

The function `scan` drives the whole process.  It maintains three variables:
- `state` is the current state of the scanner.  It starts as `PREAMBLE`.
- `stack` stores the markers that are needed to interpret a closing brace.
- `pos`   is the position where the next token starts.

The `while` loop of Python is replaced by the recursive function `loop`, which gets the current state and
position as arguments and returns the final state.  In every iteration the master pattern of the current
state is matched against the input `text` at the position `pos`.  Note that the function `Re.Pattern.match_`
only tries to find a match that starts exactly at the given position, which is precisely what a scanner needs.
We then ask the match object `mo` for the name of the group that is responsible for the match and dispatch
to the function that processes the tokens of the current state.  This function returns the state that the
scanner is in after the token has been processed.

**Function `scan text`**
- *Input:* `text` is the content of a $\LaTeX$ file.
- *Output:* `()`.  The function writes the translation of `text` into <span style="font-variant:small-caps;">Html</span> to the current output, i.e. to `!outfile`.  The exception `Syntax_error` is raised if `text` is malformed or does not contain the string `\end{document}`.

In [ ]:
let scan (text : string) : unit =
  let stack : marker Stack.t = Stack.create () in
  let rec loop (state : state) (pos : int) : state =
    if state <> DONE && pos < String.length text then
      match Re.Pattern.match_ ~pos (master_pattern state) text with
      | None -> assert false     (* the last token of every state matches any character *)
      | Some mo ->
          let token = token_of_name (Option.get (Re.lastgroup mo)) in
          let value = Re.group mo 0 in
          let state =
            match state with
            | PREAMBLE -> process_preamble token value
            | TEXT     -> process_text token value
            | FORMULA  -> process_formula token value stack
            | DONE     -> assert false     (* excluded by the condition of the loop *)
          in
          loop state (Re.end_ mo)
    else state
  in
  if loop PREAMBLE 0 <> DONE then
    raise (Syntax_error {|missing "\end{document}"|})

## Testing your Implementation

The scanner is already able to deal with square roots.  The cell below redirects the output into a
string instead of the file `example.html`, so that you can test your implementation as often as you
like without destroying the output file.  Extend the test input as you implement more tokens.

In [ ]:
let saved_outfile = !outfile                  (* remember the real output file *)
;;
outfile := Memory (Buffer.create 1000);;      (* collect the output of this test in a string *)

scan {|\documentclass{article}
\begin{document}
The square root of $a$ is $\sqrt{a}$.
\end{document}
|};;

print_endline (contents_of_outfile ());;
outfile := saved_outfile                      (* restore the real output file *)

## Running the Translator

Once you have completed both the token specification of the state `FORMULA` and the function
`process_formula`, the cell below translates the file `example.tex` that we have read into the
variable `data`.  As long as your implementation is incomplete, this cell reports an error that
tells you what is still missing:
- A `Syntax_error` complaining about an unexpected character means that a token is still missing
  from the token specification.
- If the token has been recognized, but the corresponding case in `process_formula` has not been implemented yet,
  the output is incomplete.  Moreover, a marker that is never pushed may lead to a `Syntax_error` when the
  corresponding closing brace is read.

Note that a failed run leaves an incomplete `example.html` behind, so you have to execute the cell
that opens this file again before you retry.

In [ ]:
scan data

In [ ]:
close_outfile ()

Let us inspect the <span style="font-variant:small-caps;">Html</span> file that has been produced.

In [ ]:
print_endline (read_file "example.html")

Finally, we open the file in a browser.  Again, if you are not running
<span style="font-variant:small-caps;">MacOS</span> you have to replace the command `open`.

In [ ]:
Sys.command "open example.html"

## What Happens if the Input is Malformed?

The functions defined above write their output to the sink `!outfile`.  Since the file
has already been closed, we redirect the output into a buffer, which stores everything in memory.
This way we can experiment without overwriting `example.html`.

In [ ]:
outfile := Memory (Buffer.create 1000)

The formula below contains an opening brace that is never closed.

In [ ]:
let broken = {|\documentclass{article}
\begin{document}
$$ \frac{1}{6 $$
\end{document}
|}
;;
try scan broken
with Syntax_error error -> print_endline error

The formula below contains a character that is not supported inside a formula.

In [ ]:
outfile := Memory (Buffer.create 1000);;

let broken = {|\documentclass{article}
\begin{document}
$$ a \wedge b $$
\end{document}
|}
;;
try scan broken
with Syntax_error error -> print_endline error

## Further Exercises

Once your translator works, it still supports only a small subset of $\LaTeX$.  Try to extend it:
1. Support the commands `\alpha`, `\beta`, and `\gamma`, which have to be translated into the
   <span style="font-variant:small-caps;">Html</span> entities `&alpha;`, `&beta;`, and `&gamma;`.
2. Support subscripts, i.e. expressions of the form `x_{1}`.  In
   <span style="font-variant:small-caps;">MathML</span>, a subscript is written using the tag
   `<msub>`.
3. Support the command `\sqrt` with an optional argument, i.e. expressions of the form
   `\sqrt[3]{x}`, which denote the third root of $x$.  The corresponding
   <span style="font-variant:small-caps;">MathML</span> tag is `<mroot>`.
4. In the state `TEXT` the characters `<`, `>`, and `&` are echoed unchanged.  Strictly speaking,
   these characters have to be replaced by the entities `&lt;`, `&gt;`, and `&amp;`.  Add a token
   that takes care of this.
5. Ordinary text is currently not wrapped into paragraphs.  Add a token that recognizes an empty
   line and translates it into the tag `<p>`.